<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day16_practice2_LSTM_%EA%B8%B0%EC%9A%B8%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# RNN의 약점: 문장이 길수록 '첫 단어'의 기울기 소실

In [2]:
import torch
import torch.nn as nn

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
# 왜 RNN은 기울기 소실되나 - 단어가 1000개면 tanh를 1000번 계산. 역전파로 기울기를 1000번 곱하면 0에 수렴된다
def first_word_grad(model_cls, seq_len, hidden=32):
  torch.manual_seed(0)
  model = model_cls(16, hidden, batch_first=True) # 16입력차원, 은닉 32차원, 입력 텐서 형태(배치크기, 단어길이, 벡터차원)
  x = torch.randn(1, seq_len, 16, requires_grad=True) # 문장 1개, 단어 개수, 16차원 벡터, 기울기 계산
  out, _= model(x) # 매 걸음의 출력 h들
  out[0, -1].sum().backward() # 0번 문장의 마지막 h 벡터값을 다 더해서 스칼라 숫자 하나로 backward()에 넣어준다
  return x.grad[0,0].abs().mean().item() # 0번 문장의 0번째 단어의 기울기(배치크기, 단어길이, 벡터차원)를 16개 평균내서 하나의 대표값으로

print(f"{'길이':>6s} | {'RNN':>12s} | {'LSTM':>12s}")
for L in [5, 20, 50, 100]: # 단어 개수
  g_rnn = first_word_grad(nn.RNN, L)
  g_lstm = first_word_grad(nn.LSTM, L)
  print(f"{L:>6d} | {g_rnn:>12.2e} | {g_lstm:>12.2e}") # LSTM (옵션 미설정 결과)

    길이 |          RNN |         LSTM
     5 |     2.33e-02 |     7.72e-03
    20 |     1.37e-06 |     4.20e-06
    50 |     1.65e-14 |     1.60e-12
   100 |     6.98e-28 |     9.35e-23
